# Next-Character Prediction: Bigram vs LSTM vs Transformer

In this experiment, my task is next-character prediction, which means that, given the previous characters, I predict the next character. I trained three models (Bigram, LSTM, and a small Transformer) on the same dataset and compared their performance.

## 1. Data and Vocabulary

The dataset I used is Tiny Shakespeare, an excerpt from Shakespeare's plays. It contains 1,115,394 characters in total and 65 unique characters, and it is processed at the character level (by characters rather than words). For the data split, I used the first 90% for training and the last 10% for validation. Regarding the download instructions, running the first code cell below will automatically download the file from https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-11-02 21:08:55--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-11-02 21:08:55 (36.1 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
import torch

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(len(text))    # total number of characters
print(text[:200])   # peek at the beginning

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    """Convert a string into a list of integer IDs."""
    return [stoi[c] for c in s]

def decode(ids):
    """Convert a list of integer IDs back into a string."""
    return ''.join([itos[i] for i in ids])

data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# quick checks
print("vocab size:", vocab_size)
print(encode("hello"))
print(decode(encode("hello")))
print(len(train_data), len(val_data))

vocab size: 65
[46, 43, 50, 50, 53]
hello
1003854 111540


## 2. Shared Training Setup

To make sure the comparison is fair, all three models use the same dataset and train/validation split, and they also share the same `get_batch` and `train_model` functions with the same batch size and training steps. They are all trained and timed in the same CPU environment, and the parameter counts of LSTM and Transformer are also very close (around 112,000). The only exception is the learning rate: Bigram uses a higher learning rate of 1e-2 while LSTM and Transformer use 1e-3, because Bigram did not converge within 3000 steps at 1e-3.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)

block_size = 64   # characters of context per example
batch_size = 32   # examples per training step

def get_batch(split):
    """Sample random chunks (x) and their next-character targets (y)."""
    source = train_data if split == 'train' else val_data
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in ix])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print(device)
print(repr(decode(xb[0][:20].tolist())))
print(repr(decode(yb[0][:20].tolist())))

cpu
'Paris!\nWhat said my '
'aris!\nWhat said my m'


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import time

@torch.no_grad()
def estimate_loss(model, eval_iters=100):
    """Average loss on train and val splits over several random batches."""
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

def train_model(model, max_iters=3000, lr=1e-3, eval_interval=500):
    """Train any model with the same loop; return final losses and elapsed time."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    start = time.time()
    for step in range(max_iters):
        if step % eval_interval == 0:
            losses = estimate_loss(model)
            print(f"step {step}: train {losses['train']:.4f}, val {losses['val']:.4f}")
        xb, yb = get_batch('train')
        _, loss = model(xb, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    elapsed = time.time() - start
    losses = estimate_loss(model)
    print(f"final: train {losses['train']:.4f}, val {losses['val']:.4f}, time {elapsed:.1f}s")
    return losses, elapsed

@torch.no_grad()
def generate(model, max_new_tokens=300):
    """Generate text by repeatedly sampling the next character."""
    model.eval()
    idx = torch.zeros((1, 1), dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -block_size:])
        probs = F.softmax(logits[:, -1, :], dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    model.train()
    return decode(idx[0].tolist())

## 3. Model 1: Bigram

It is a 65 x 65 lookup table that uses the current character to find scores for every possible next character. However, it only looks at the current character and ignores all earlier characters.

In [ ]:
class BigramModel(nn.Module):
    """Predict the next character using only the current character."""
    def __init__(self, vocab_size):
        super().__init__()
        # rows: current character, columns: score for each possible next character
        self.table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.table(idx)
        if targets is None:
            return logits, None
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

In [ ]:
bigram = BigramModel(vocab_size).to(device)
bigram_losses, bigram_time = train_model(bigram, lr=1e-2)
print(generate(bigram))

step 0: train 4.6507, val 4.6572
step 500: train 2.5240, val 2.5486
step 1000: train 2.4753, val 2.4999
step 1500: train 2.4675, val 2.4912
step 2000: train 2.4644, val 2.4897
step 2500: train 2.4595, val 2.4916
final: train 2.4556, val 2.4854, time 12.9s

AI s icor edon s n y the,
YORERYongrithie yorrst t thys,
ARCounte, of her houkn t ak, an?
He sis serin matisunthe;
jes tichyof t fastingrtiche h:
NGLE wh ivee imyorked y
I: witcio u, oy w,

RIntoot Why trt rre

PO t.
EYoury did wasume s ch.
Thavesu,
Ay Whinore IXENCK:
KI,
We r thourr, h wilisery tou


## 4. Model 2: LSTM

It keeps track of context like taking notes in a notebook. It reads character by character and updates the notebook at each step. The longer the sequence it reads, the weaker the earlier information becomes, though it still captures more previous context than Bigram.

In [ ]:
class LSTMModel(nn.Module):
    """Predict the next character with an LSTM that carries memory across the sequence."""
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, idx, targets=None):
        x = self.embed(idx)        # (batch, time, embed_dim)
        out, _ = self.lstm(x)      # (batch, time, hidden_dim)
        logits = self.head(out)    # (batch, time, vocab_size)
        if targets is None:
            return logits, None
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

In [ ]:
lstm = LSTMModel(vocab_size).to(device)
print("parameters:", sum(p.numel() for p in lstm.parameters()))
lstm_losses, lstm_time = train_model(lstm)
print(generate(lstm))

parameters: 111873
step 0: train 4.1942, val 4.1943
step 500: train 2.0025, val 2.0415
step 1000: train 1.8096, val 1.9107
step 1500: train 1.7198, val 1.8386
step 2000: train 1.6630, val 1.8000
step 2500: train 1.6191, val 1.7641
final: train 1.5930, val 1.7454, time 170.6s

I'll took or to himselfllenuled to the hand his mativelion and good!

KING RICHARD III:
My gove hash to meanself.
Shoush, ast the kings, gos must seal cratorant to mine, for I had-Jast dayise
belend, hour susing till the good thoughterfur; and fromber's
I grace it: well has your and towfard!

HENRY 


## 5. Model 3: Small Transformer

 Each character can look back at any earlier character without being affected by distance. Every character has three vectors: Query (what to look for), Key (its own label), and Value (the actual content to share). It also uses a causal mask so each character can only see itself and preceding characters, preventing it from cheating and looking ahead at the answer. The model consists of five main components:

* Token embeddings: Assigns a feature card to each unique character.

* Positional information: Assigns a seat number card to each position so the model knows the order of the characters.

* Self-attention: Allows characters to exchange information and look back at earlier characters.

* Feedforward layer: Lets each character process and digest the information after gathering it.

* Output layer: Assigns a score to each of the 65 characters to predict the next character.

The transformer architecture in this notebook is based on Andrej Karpathy’s "Let’s build GPT" tutorial.


In [ ]:
n_embd = 64                  # size of each character's feature vector
n_head = 4                   # number of attention heads
n_layer = 2                  # number of transformer blocks
head_size = n_embd // n_head

class Head(nn.Module):
    """One head of causal self-attention."""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        # how well each query matches each key
        scores = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        # block future positions
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        v = self.value(x)
        return weights @ v

In [ ]:
class MultiHeadAttention(nn.Module):
    """Several attention heads in parallel, then combine their outputs."""
    def __init__(self, n_head, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

class FeedForward(nn.Module):
    """Small per-position network applied after attention."""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """Transformer block: attention (look at other characters), then feedforward (process)."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.attn = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # residual connection
        x = x + self.ffwd(self.ln2(x))   # residual connection
        return x

In [ ]:
class TransformerModel(nn.Module):
    """Small character-level transformer language model."""
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, n_embd)
        self.pos_embed = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embed(idx)
        pos = self.pos_embed(torch.arange(T, device=idx.device))
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        if targets is None:
            return logits, None
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

In [ ]:
transformer = TransformerModel(vocab_size).to(device)
print("parameters:", sum(p.numel() for p in transformer.parameters()))
transformer_losses, transformer_time = train_model(transformer)
print(generate(transformer))

parameters: 112193
step 0: train 4.3243, val 4.3159
step 500: train 2.2757, val 2.2949
step 1000: train 2.0359, val 2.0852
step 1500: train 1.8921, val 1.9909
step 2000: train 1.8010, val 1.9209
step 2500: train 1.7449, val 1.8924
final: train 1.7011, val 1.8543, time 235.8s


CAMILLO:
Yourself held.
That you true sovy for man,
And this kio apted the sapiefore.

PEUCBER:
The do peacion centakes; cause: you to trumines begg
Of nothing, and actio shall his me
Deach head the who confe im blown:

Provother; Let one pony
teo my gaill-coldagerfort lify he Guntlencan non the
wi


## 6. Results and Comparison

In the next two cells, the first one runs the extended 6000-step training experiment, and the second one shows the summary table of my results. The full analysis and comparison can be found in the README.

In [ ]:
# Extended training: all three models trained for the same longer schedule
torch.manual_seed(42)
bigram_long = BigramModel(vocab_size).to(device)
bigram_long_losses, bigram_long_time = train_model(bigram_long, max_iters=6000, lr=1e-2)

torch.manual_seed(42)
lstm_long = LSTMModel(vocab_size).to(device)
lstm_long_losses, lstm_long_time = train_model(lstm_long, max_iters=6000)
print(generate(lstm_long))

torch.manual_seed(42)
transformer_long = TransformerModel(vocab_size).to(device)
transformer_long_losses, transformer_long_time = train_model(transformer_long, max_iters=6000)
print(generate(transformer_long))

step 0: train 4.7537, val 4.7627
step 500: train 2.5263, val 2.5541
step 1000: train 2.4757, val 2.5019
step 1500: train 2.4675, val 2.4921
step 2000: train 2.4651, val 2.4901
step 2500: train 2.4591, val 2.4923
step 3000: train 2.4560, val 2.4859
step 3500: train 2.4550, val 2.4878
step 4000: train 2.4536, val 2.4821
step 4500: train 2.4524, val 2.4800
step 5000: train 2.4528, val 2.4808
step 5500: train 2.4534, val 2.4843
final: train 2.4540, val 2.4841, time 24.5s
step 0: train 4.1902, val 4.1899
step 500: train 2.0162, val 2.0523
step 1000: train 1.8276, val 1.9204
step 1500: train 1.7310, val 1.8514
step 2000: train 1.6652, val 1.8037
step 2500: train 1.6271, val 1.7627
step 3000: train 1.6018, val 1.7407
step 3500: train 1.5725, val 1.7329
step 4000: train 1.5559, val 1.7148
step 4500: train 1.5378, val 1.7077
step 5000: train 1.5262, val 1.6917
step 5500: train 1.5141, val 1.6808
final: train 1.5089, val 1.6739, time 335.0s

For, with us the lick calows, all wile and his,
Is pra

In [ ]:
import pandas as pd

rows = [
    ('Bigram', 3000, bigram, bigram_losses, bigram_time),
    ('LSTM', 3000, lstm, lstm_losses, lstm_time),
    ('Transformer', 3000, transformer, transformer_losses, transformer_time),
    ('Bigram', 6000, bigram_long, bigram_long_losses, bigram_long_time),
    ('LSTM', 6000, lstm_long, lstm_long_losses, lstm_long_time),
    ('Transformer', 6000, transformer_long, transformer_long_losses, transformer_long_time),
]

results = pd.DataFrame([
    {
        'Model': name,
        'Steps': steps,
        'Parameters': sum(p.numel() for p in model.parameters()),
        'Train Loss': round(losses['train'], 4),
        'Val Loss': round(losses['val'], 4),
        'Time (s)': round(t, 1),
    }
    for name, steps, model, losses, t in rows
])
results

,Model,Steps,Parameters,Train Loss,Val Loss,Time (s)
0,Bigram,3000,4225,2.4556,2.4854,12.9
1,LSTM,3000,111873,1.5930,1.7454,170.6
2,Transformer,3000,112193,1.7011,1.8543,235.8
3,Bigram,6000,4225,2.4540,2.4841,24.5
4,LSTM,6000,111873,1.5089,1.6739,335.0
5,Transformer,6000,112193,1.5897,1.7573,470.4
